# Per-gene Spearman correlation: W (UTR-length expected rank) vs lognorm UMI expression, Knoblich human organoids

## Aim

Within each pseudotime lineage (Excitatory + Inhibitory) of the Knoblich CRISPR
organoid dataset, we test whether per-cell UTR-length composition is coupled to
per-cell expression of the same gene, and whether the coupling shifts under CRISPR
perturbation of 36 ASD-risk TFs (induced vs non-induced).

## Quantities

- `W_c` (cell c, gene g): expected normalised UTR-length rank
  `W_c = sum_k r_k * pi_k(c)`, `r_k = (rankdata(utr_size_k) - 1) / (K - 1)`
- `expr_c`: log-normalised UMI expression (recomputed from .raw, full ~43k genes)

## Methodological choices

1. UTR proportions are unsmoothed (per-cell raw site_count / gene_total).
2. Expression: `normalize_total(target_sum=1e4) + log1p` recomputed from `.raw`.
3. Per lineage: marginal Spearman + within-celltype median (>=10 cells/CT) +
   library-size diagnostic.
4. Treatment split: within each lineage compute Spearman separately within
   `induced=True` cells and within `induced=False` cells; `delta_rho_treatment
= rho_induced - rho_noninduced` flags genes where the coupling shifts.
5. ASD annotation: overlay flags from `asd_significant_in_knoblich.csv` (63
   curated SFARI+CHOOSE genes) on volcanos and exemplar panels.

## Output

- Per-lineage main CSV: `correlation_analyses_expression/utr_W_expression_spearman_per_gene_unsmoothed_<lineage>.csv`
- Per-lineage treatment CSV (joined into main CSV; same file)
- Figures: `correlation_analyses_expression/figures/<plot>_<lineage>.{png,pdf}`

## Caveats (see final cell)

- `induced` is constant within each sample (14 induced samples, 8 non-induced).
  Super_batch 137 = 100% induced. Treatment-split rho is exploratory; the cross-
  comparison with the treatment DRIMSeq LRT is the safer outer signal.
- Two lineages analysed independently (cells are mutually exclusive).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
from collections import defaultdict
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.sparse import issparse, csr_matrix, csc_matrix
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
COL_SHORT = "#2166AC"
COL_LONG = "#B2182B"
COL_NS = "lightgray"
COL_ASD = "#F39C12"  # ASD-flagged genes
COL_TREAT = "#8E44AD"  # treatment-LRT-sig genes
SIG_THR = 0.05
RHO_VIS_THR = 0.3

In [ ]:
BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
EXPR_PATH = (
    BASEPATH / "cellranger" / "aggregated_annotated_fix_filtered_normalized_dpt.h5ad"
)
UTR_RAW_PATH = (
    BASEPATH / "utrcalling" / "merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"
)
ASD_CSV = BASEPATH / "utrcalling" / "asd_results" / "asd_significant_in_knoblich.csv"

EFFECT_SIZES_DIR = BASEPATH / "utrcalling" / "effect_sizes"
DIRMULT_DIR = BASEPATH / "utrcalling" / "dirmult_results"
COMPOSITE_DIR = BASEPATH / "utrcalling" / "composite_ranking"
TREATMENT_DIR = BASEPATH / "utrcalling" / "dirmult_treatment_results"

STOREPATH = BASEPATH / "utrcalling" / "correlation_analyses_expression"
FIGDIR = STOREPATH / "figures"
os.makedirs(STOREPATH, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

MIN_GENE_COUNTS = 1
MIN_EXPRESSING_CELLS = 30
MIN_CELLS_PER_CT = 10
MIN_TREATMENT_CELLS = 30
PROGRESS_EVERY = 1000
SMOKE_TEST_N = 100

LINEAGES = [
    ("Excitatory neurons", "excitatory", "dpt_pseudotime_excitatory", "EXC"),
    ("Inhibitory neurons", "inhibitory", "dpt_pseudotime_inhibitory", "INH"),
]


def savefig(name):
    fig = plt.gcf()
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", bbox_inches="tight", dpi=300)


print("output dir:", STOREPATH)
print("figures dir:", FIGDIR)

## Step 1: load UTR raw counts + expression `.raw` + cell-ID reconciliation

UTR uses `BARCODE-1_BATCH`, expression uses `BARCODE-BATCH`. Reconcile via
(barcode, batch_int) keys extracted from obs_names; intersection should be
71,561 cells. We keep the UTR matrix in CSC sparse format throughout (full
densify would be ~35 GB float64).


In [ ]:
adata_utr = sc.read_h5ad(UTR_RAW_PATH)
print("UTR raw:", adata_utr.shape, "X type:", type(adata_utr.X).__name__)

adata_expr_full = sc.read_h5ad(EXPR_PATH)
print("expression:", adata_expr_full.shape, "| .raw:", adata_expr_full.raw.shape)

adata_expr = adata_expr_full.raw.to_adata()
adata_expr.obs = adata_expr_full.obs.copy()
total_counts_obs = adata_expr_full.obs["total_counts"].copy()
del adata_expr_full


def _utr_key(name):
    bc, suf = name.split("-1_", 1)
    return (bc, int(suf))


def _expr_key(name):
    bc, suf = name.rsplit("-", 1)
    return (bc, int(suf))


utr_keys = [_utr_key(n) for n in adata_utr.obs_names]
expr_keys = [_expr_key(n) for n in adata_expr.obs_names]
expr_key_to_idx = {k: i for i, k in enumerate(expr_keys)}

utr_in_expr = [k in expr_key_to_idx for k in utr_keys]
keep_utr_idx = np.where(utr_in_expr)[0]
keep_expr_idx = np.array(
    [expr_key_to_idx[utr_keys[i]] for i in keep_utr_idx], dtype=np.int64
)
print(f"UTR cells in expression: {len(keep_utr_idx)} (of {len(utr_keys)})")
print(f"expression cells dropped (not in UTR): {len(expr_keys) - len(keep_utr_idx)}")

adata_utr = adata_utr[keep_utr_idx].copy()
adata_expr = adata_expr[keep_expr_idx].copy()
# Align obs_names to UTR convention so a shared index works
adata_expr.obs_names = adata_utr.obs_names.copy()
total_counts_obs = total_counts_obs.iloc[keep_expr_idx].copy()
total_counts_obs.index = adata_utr.obs_names
assert list(adata_utr.obs_names) == list(adata_expr.obs_names)
print("aligned:", adata_utr.n_obs, "cells")

In [ ]:
print("normalize_total(target_sum=1e4) + log1p on expression .raw ...")
sc.pp.normalize_total(adata_expr, target_sum=1e4)
sc.pp.log1p(adata_expr)
if issparse(adata_expr.X):
    nnz = adata_expr.X.nnz
    tot = adata_expr.X.shape[0] * adata_expr.X.shape[1]
    print(f"lognorm sparse: nnz={nnz:,}, density={nnz/tot:.3%}")
else:
    print("lognorm dense, shape", adata_expr.X.shape)
print(
    f"value range: [{float(adata_expr.X.min()):.3f}, {float(adata_expr.X.max()):.3f}]"
)

In [ ]:
log_total_counts_all = np.log1p(total_counts_obs.values.astype(float))
print(
    f"log(total_counts+1) mean: {log_total_counts_all.mean():.2f}, range [{log_total_counts_all.min():.2f}, {log_total_counts_all.max():.2f}]"
)

# Convert UTR matrix to CSC for fast column slicing
print("converting UTR matrix to CSC ...")
if issparse(adata_utr.X):
    X_utr_csc = adata_utr.X.tocsc().astype(np.float64)
else:
    X_utr_csc = csc_matrix(np.asarray(adata_utr.X, dtype=np.float64))
print(
    f"X_utr CSC nnz: {X_utr_csc.nnz:,}, density {X_utr_csc.nnz/(X_utr_csc.shape[0]*X_utr_csc.shape[1]):.3%}"
)

X_expr_csr = adata_expr.X if issparse(adata_expr.X) else csr_matrix(adata_expr.X)
# Also keep a CSC view for fast column slicing
X_expr_csc = X_expr_csr.tocsc()
print(f"X_expr CSR nnz: {X_expr_csr.nnz:,}")

## Step 2: build ENSEMBL <-> symbol gene map (multi-site only)

UTR `var['name']` is ENSEMBL, `var['gene_name']` is symbol. Expression
`var_names` are symbols only. We:

1. Group UTR sites by ENSEMBL `name`.
2. Keep K >= 2 sites.
3. Drop ENSEMBL if symbol absent from `adata_expr.var_names`.
4. For symbols mapping to multiple ENSEMBLs, keep only the ENSEMBL with the
   most sites (most-informative variant).


In [ ]:
var_names_utr = list(adata_utr.var_names)
var_name_to_idx = {v: i for i, v in enumerate(var_names_utr)}

# Build gene_id -> list of (site_idx, utr_size, gene_name)
gene_to_sites = defaultdict(list)
for i, (site_name, row) in enumerate(adata_utr.var.iterrows()):
    gene_id = row["name"]  # ENSEMBL
    gene_to_sites[gene_id].append(
        (var_name_to_idx[site_name], float(row["utr_size"]), row["gene_name"])
    )

# Restrict to multi-site
multi_site = {gid: sites for gid, sites in gene_to_sites.items() if len(sites) >= 2}
print(f"genes (any K): {len(gene_to_sites)}; multi-site (K>=2): {len(multi_site)}")

# Resolve symbol -> ENSEMBL ambiguity: keep ENSEMBL with most sites
expr_symbols = set(adata_expr.var_names)
expr_sym_to_idx = {s: i for i, s in enumerate(adata_expr.var_names)}

symbol_to_candidates = defaultdict(list)
for gid, sites in multi_site.items():
    symbol = sites[0][2]
    symbol_to_candidates[symbol].append((gid, len(sites)))

# Drop missing-symbol; pick best ENSEMBL per symbol
kept_gids = set()
n_no_symbol = 0
n_ambiguous = 0
for symbol, cands in symbol_to_candidates.items():
    if symbol not in expr_symbols:
        n_no_symbol += len(cands)
        continue
    if len(cands) > 1:
        n_ambiguous += len(cands) - 1
        best_gid = max(cands, key=lambda x: x[1])[0]
    else:
        best_gid = cands[0][0]
    kept_gids.add(best_gid)

print(f"dropped (no symbol in expression): {n_no_symbol}")
print(f"dropped (symbol mapped to multiple ENSEMBL, kept best): {n_ambiguous}")
print(f"kept multi-site genes: {len(kept_gids)}")

# Build gene_meta tuples
gene_meta = []
for gid in kept_gids:
    sites = multi_site[gid]
    site_idxs = np.array([s[0] for s in sites], dtype=np.int64)
    utr_sizes = np.array([s[1] for s in sites], dtype=np.float64)
    symbol = sites[0][2]
    K = len(sites)
    ranks = (rankdata(utr_sizes, method="average") - 1.0) / (K - 1)
    expr_idx = expr_sym_to_idx[symbol]
    gene_meta.append((gid, symbol, site_idxs, K, ranks, expr_idx))

print(f"final gene_meta entries: {len(gene_meta)}")

## Step 3: load ASD reference for plot annotation


In [ ]:
asd_df = pd.read_csv(ASD_CSV)
print("ASD reference shape:", asd_df.shape)
print("ASD columns:", list(asd_df.columns))
asd_lookup = {}
for _, row in asd_df.iterrows():
    asd_lookup[row["gene_id"]] = {
        "sfari_score": row.get("sfari_score", np.nan),
        "is_sfari_high": bool(row.get("is_sfari_high", False)),
        "is_li2023": bool(row.get("is_li2023", False)),
        "is_asd": bool(row.get("is_asd", False)),
        "gene_name": row.get("gene_name", ""),
    }
print(f"ASD lookup entries: {len(asd_lookup)}")

## Step 4: per-gene metrics function + smoke test (EXC, 100 genes)


In [ ]:
def per_gene_metrics(
    gene_id,
    gene_name,
    site_idxs,
    K,
    ranks,
    expr_col_idx,
    X_utr_csc,
    X_expr_csr,
    log_libsize,
    celltype_codes,
    celltype_categories,
    induced_arr=None,
):
    """Compute per-gene metrics for current lineage subset. All input arrays are
    already lineage-restricted by the caller."""
    sub_sparse = X_utr_csc[:, site_idxs]
    gene_total = np.asarray(sub_sparse.sum(axis=1)).ravel()
    cells_include = gene_total >= MIN_GENE_COUNTS
    n_inc = int(cells_include.sum())
    row = dict(
        gene_id=gene_id,
        gene_name=gene_name,
        n_sites=K,
        n_expressing_cells=n_inc,
        rho_W_expr=np.nan,
        pvalue=np.nan,
        rho_W_expr_celltype_median=np.nan,
        n_celltypes_used=0,
        rho_W_expr_per_celltype_min=np.nan,
        rho_W_expr_per_celltype_max=np.nan,
        mean_lognorm_expr=np.nan,
        rho_W_libsize=np.nan,
        rho_induced=np.nan,
        pvalue_induced=np.nan,
        n_induced_cells_used=0,
        rho_noninduced=np.nan,
        pvalue_noninduced=np.nan,
        n_noninduced_cells_used=0,
        delta_rho_treatment=np.nan,
    )
    if n_inc < MIN_EXPRESSING_CELLS:
        return row

    sub_inc = sub_sparse[cells_include].toarray()
    tot_inc = gene_total[cells_include]
    props = sub_inc / tot_inc[:, None]
    W_c = props @ ranks

    expr_col = X_expr_csr[:, expr_col_idx]
    if issparse(expr_col):
        expr_col = np.asarray(expr_col.todense()).ravel()
    expr_inc = expr_col[cells_include]
    row["mean_lognorm_expr"] = float(expr_inc.mean())

    if W_c.std() > 0 and expr_inc.std() > 0:
        rho_marg, p_marg = spearmanr(W_c, expr_inc)
        row["rho_W_expr"] = float(rho_marg)
        row["pvalue"] = float(p_marg)

    libsize_inc = log_libsize[cells_include]
    if W_c.std() > 0 and libsize_inc.std() > 0:
        rho_lib, _ = spearmanr(W_c, libsize_inc)
        row["rho_W_libsize"] = float(rho_lib)

    ct_inc = celltype_codes[cells_include]
    per_ct = []
    for code_ in range(len(celltype_categories)):
        m = ct_inc == code_
        if int(m.sum()) >= MIN_CELLS_PER_CT:
            w_ct = W_c[m]
            e_ct = expr_inc[m]
            if w_ct.std() > 0 and e_ct.std() > 0:
                rho_ct, _ = spearmanr(w_ct, e_ct)
                per_ct.append(float(rho_ct))
    if per_ct:
        row["rho_W_expr_celltype_median"] = float(np.median(per_ct))
        row["rho_W_expr_per_celltype_min"] = float(np.min(per_ct))
        row["rho_W_expr_per_celltype_max"] = float(np.max(per_ct))
        row["n_celltypes_used"] = int(len(per_ct))

    if induced_arr is not None:
        ind_inc = induced_arr[cells_include]
        for flag, key_rho, key_p, key_n in [
            (True, "rho_induced", "pvalue_induced", "n_induced_cells_used"),
            (False, "rho_noninduced", "pvalue_noninduced", "n_noninduced_cells_used"),
        ]:
            m = ind_inc == flag
            n_grp = int(m.sum())
            row[key_n] = n_grp
            if n_grp >= MIN_TREATMENT_CELLS:
                w_g = W_c[m]
                e_g = expr_inc[m]
                if w_g.std() > 0 and e_g.std() > 0:
                    rho_g, p_g = spearmanr(w_g, e_g)
                    row[key_rho] = float(rho_g)
                    row[key_p] = float(p_g)
        if not np.isnan(row["rho_induced"]) and not np.isnan(row["rho_noninduced"]):
            row["delta_rho_treatment"] = row["rho_induced"] - row["rho_noninduced"]
    return row


def _bh(pvals):
    out = np.full(len(pvals), np.nan)
    valid = ~np.isnan(pvals)
    if int(valid.sum()) > 0:
        _, padj_valid, _, _ = multipletests(pvals[valid], method="fdr_bh")
        out[valid] = padj_valid
    return out


def process_lineage(lineage_name, pt_col, n_smoke=None):
    """Subset to lineage and compute per-gene metrics. If n_smoke given, only run
    that many random genes (for smoke testing)."""
    obs = adata_utr.obs
    mask = (obs["lineage"] == lineage_name) & obs[pt_col].notna()
    cell_idx = np.where(mask.values)[0]
    print(f"--- lineage {lineage_name}: {len(cell_idx)} cells ---")

    X_utr_lin = X_utr_csc[cell_idx, :].tocsc()
    X_expr_lin = X_expr_csc[cell_idx, :]
    log_lib_lin = log_total_counts_all[cell_idx]
    induced_lin = obs["induced"].values[cell_idx]

    # Within-lineage celltype categories (drop empty)
    ct_series = obs["celltype"].iloc[cell_idx].astype(str)
    ct_present = [
        c for c in ct_series.unique() if (ct_series == c).sum() >= MIN_CELLS_PER_CT
    ]
    ct_present = sorted(ct_present)
    celltype_codes = pd.Categorical(ct_series, categories=ct_present).codes
    print(f"  celltypes (>=10 cells): {ct_present}")

    iter_idx = list(range(len(gene_meta)))
    if n_smoke is not None:
        rng = np.random.default_rng(0)
        iter_idx = rng.choice(
            len(gene_meta), size=min(n_smoke, len(gene_meta)), replace=False
        ).tolist()

    rows = []
    for j, gi in enumerate(iter_idx):
        if (j + 1) % PROGRESS_EVERY == 0 or j == 0:
            print(f"  gene {j+1}/{len(iter_idx)}")
        tup = gene_meta[gi]
        rows.append(
            per_gene_metrics(
                *tup,
                X_utr_csc=X_utr_lin,
                X_expr_csr=X_expr_lin,
                log_libsize=log_lib_lin,
                celltype_codes=celltype_codes,
                celltype_categories=ct_present,
                induced_arr=induced_lin,
            )
        )
    df = pd.DataFrame(rows)
    df["padj_BH"] = _bh(df["pvalue"].values)
    df["padj_BH_induced"] = _bh(df["pvalue_induced"].values)
    df["padj_BH_noninduced"] = _bh(df["pvalue_noninduced"].values)
    df["abs_rho_W_expr"] = df["rho_W_expr"].abs()
    df["abs_delta_rho_treatment"] = df["delta_rho_treatment"].abs()

    # ASD annotation
    df["sfari_score"] = df["gene_id"].map(
        lambda g: asd_lookup.get(g, {}).get("sfari_score", np.nan)
    )
    df["is_sfari_high"] = df["gene_id"].map(
        lambda g: asd_lookup.get(g, {}).get("is_sfari_high", False)
    )
    df["is_li2023"] = df["gene_id"].map(
        lambda g: asd_lookup.get(g, {}).get("is_li2023", False)
    )
    df["is_asd"] = df["gene_id"].map(
        lambda g: asd_lookup.get(g, {}).get("is_asd", False)
    )
    df["asd_any"] = df["is_sfari_high"] | df["is_li2023"] | df["is_asd"]
    return df, ct_present


# Smoke test on EXC
print("=== smoke test on EXC, 100 genes ===")
smoke_exc, _ = process_lineage(
    "Excitatory neurons", "dpt_pseudotime_excitatory", n_smoke=SMOKE_TEST_N
)
print(
    f"smoke EXC: {len(smoke_exc)} genes, non-NaN rho: {smoke_exc['rho_W_expr'].notna().sum()}"
)
print(smoke_exc.head(5))

## Step 5: full run, EXC


In [ ]:
import time

t0 = time.time()
df_EXC, ct_EXC = process_lineage("Excitatory neurons", "dpt_pseudotime_excitatory")
print(f"EXC runtime: {time.time()-t0:.1f}s")
print(
    f"EXC total rows: {len(df_EXC)}; non-NaN rho: {df_EXC['rho_W_expr'].notna().sum()}"
)
sig_EXC = df_EXC["padj_BH"] < SIG_THR
print(f"EXC padj<0.05: {int(sig_EXC.sum())}")

In [ ]:
# Merge per-gene cross-comparison tables for EXC
def _load_lrt(lineage_dirname, pt_col_name):
    p = (
        DIRMULT_DIR
        / lineage_dirname
        / "controls_only_batch_number"
        / f"dirmult_regression_optimized2_joint_df5_pt-{pt_col_name}_cov-batch_number.csv"
    )
    df = pd.read_csv(p)
    return df.rename(columns={"utr_name": "gene_id", "padj": "lrt_padj_pseudotime"})[
        ["gene_id", "lrt_padj_pseudotime"]
    ]


def _load_delta_psi(lineage_dirname):
    p = EFFECT_SIZES_DIR / lineage_dirname / "delta_psi_per_gene_df5_q0.1-0.9.csv"
    return pd.read_csv(p)[["gene_id", "delta_psi", "nmi"]]


def _load_rho_W(lineage_dirname):
    p = EFFECT_SIZES_DIR / lineage_dirname / "rho_W_per_gene.csv"
    df = pd.read_csv(p)
    return df.rename(
        columns={"rho_W": "rho_W_pseudotime", "padj_BH": "padj_BH_pseudotime"}
    )[["gene_id", "rho_W_pseudotime", "padj_BH_pseudotime"]]


def _load_composite(lineage_dirname):
    p = COMPOSITE_DIR / f"composite_ranking_optimized2_{lineage_dirname}.csv"
    df = pd.read_csv(p)
    keep = [
        c
        for c in ["gene_id", "composite_score_signed", "trajectory_pattern"]
        if c in df.columns
    ]
    return df[keep]


def _load_treatment(lineage_dirname):
    """Treatment DRIMSeq CSV's 'gene_id' column actually contains gene symbols,
    not ENSEMBL. Rename and return on gene_name so the merge joins on symbol."""
    p = (
        TREATMENT_DIR
        / lineage_dirname
        / "main_only"
        / f"drimseq_gene_{lineage_dirname}_treatment_main.csv"
    )
    df = pd.read_csv(p)
    return df.rename(
        columns={"adj_pvalue": "lrt_padj_treatment", "gene_id": "gene_name"}
    )[["gene_name", "lrt_padj_treatment", "delta_psi_treatment"]]


def attach_cross(df, lineage_dirname, pt_col_name):
    lrt = _load_lrt(lineage_dirname, pt_col_name)
    print(f"  LRT rows: {len(lrt)}")
    dps = _load_delta_psi(lineage_dirname)
    print(f"  delta_psi rows: {len(dps)}")
    rho = _load_rho_W(lineage_dirname)
    print(f"  rho_W rows: {len(rho)}")
    com = _load_composite(lineage_dirname)
    print(f"  composite rows: {len(com)}")
    trt = _load_treatment(lineage_dirname)
    print(f"  treatment rows: {len(trt)}")
    for tab in (lrt, dps, rho, com):
        df = df.merge(tab, on="gene_id", how="left")
    df = df.merge(trt, on="gene_name", how="left")
    return df


print("merging cross-comparison tables for EXC ...")
df_EXC = attach_cross(df_EXC, "excitatory", "dpt_pseudotime_excitatory")
print("EXC after merges:", df_EXC.shape)
print("  with LRT padj:", df_EXC["lrt_padj_pseudotime"].notna().sum())
print("  with delta_psi:", df_EXC["delta_psi"].notna().sum())
print("  with rho_W_pseudotime:", df_EXC["rho_W_pseudotime"].notna().sum())
print(
    "  with composite_score_signed:",
    (
        df_EXC["composite_score_signed"].notna().sum()
        if "composite_score_signed" in df_EXC.columns
        else 0
    ),
)
print("  with treatment LRT padj:", df_EXC["lrt_padj_treatment"].notna().sum())

df_EXC = df_EXC.sort_values(
    "abs_rho_W_expr", ascending=False, na_position="last"
).reset_index(drop=True)
outpath_EXC = STOREPATH / "utr_W_expression_spearman_per_gene_unsmoothed_excitatory.csv"
df_EXC.to_csv(outpath_EXC, index=False)
print(f"wrote {outpath_EXC} ({len(df_EXC)} rows)")

## Step 6: full run, INH


In [ ]:
t0 = time.time()
df_INH, ct_INH = process_lineage("Inhibitory neurons", "dpt_pseudotime_inhibitory")
print(f"INH runtime: {time.time()-t0:.1f}s")
print(
    f"INH total rows: {len(df_INH)}; non-NaN rho: {df_INH['rho_W_expr'].notna().sum()}"
)
sig_INH = df_INH["padj_BH"] < SIG_THR
print(f"INH padj<0.05: {int(sig_INH.sum())}")

print("merging cross-comparison tables for INH ...")
df_INH = attach_cross(df_INH, "inhibitory", "dpt_pseudotime_inhibitory")
print("INH after merges:", df_INH.shape)

df_INH = df_INH.sort_values(
    "abs_rho_W_expr", ascending=False, na_position="last"
).reset_index(drop=True)
outpath_INH = STOREPATH / "utr_W_expression_spearman_per_gene_unsmoothed_inhibitory.csv"
df_INH.to_csv(outpath_INH, index=False)
print(f"wrote {outpath_INH} ({len(df_INH)} rows)")

## Plot guide (applied identically to Excitatory and Inhibitory lineages)

The plotting functions defined in the next cells produce three families of plots per lineage (filenames suffixed `_EXC` or `_INH`), plus a final cross-lineage plot. Read this guide once; the same panels are re-rendered for the other lineage with the same axes and colours.

### Colour conventions (consistent throughout)

- `#2166AC` (blue) = shortening direction / negative rho
- `#B2182B` (red) = lengthening direction / positive rho
- `lightgray` = not significant
- `firebrick` = significant in histograms
- `#F39C12` (orange) = ASD-flagged gene (overlay markers, star symbol, from `asd_significant_in_knoblich.csv`)
- `#8E44AD` (purple) = treatment-LRT-significant gene (overlay)
- celltype colours: matplotlib tab10 palette over the lineage-specific celltypes

---

### Diagnostic plots (7 per lineage)

Diag 1: n_expressing_cells histogram

- x: per gene, the number of lineage cells with at least one UTR-site UMI for that gene
- y: number of genes per bin
- red dashed line at x=30 marks the `MIN_EXPRESSING_CELLS` threshold; genes below are reported as NaN
- check: bulk well above 30

Diag 2: cumulative distribution of K (sites per gene)

- x: K (sites per gene); y: cumulative fraction of retained genes with K <= x
- K=2 is the most common; long tail of complex multi-site genes

Diag 3: six example gene scatters (top 3 high-coverage, bottom 3 low-coverage)

- per panel: x = W_c (0 = predominantly proximal/short, 1 = predominantly distal/long); y = lognorm UMI expression
- point colour = celltype within the lineage
- title = gene name, K, n_expressing_cells, rho

Diag 4: histogram of `rho_W_libsize`

- x: per-gene Spearman(W_c, log(total UMI counts in the cell))
- y: number of genes
- peaks near +/-1 = library-size confounding (warning); near 0 = clean

Diag 5: W per celltype violin

- x: lineage celltypes; y: W_c pooled over the top-100 high-confidence genes (sig + |rho|>0.3)
- shows whether W shifts along the developmental order of the lineage's celltypes

Diag 6: detection vs expression

- x: gene's mean lognorm expression; y: n_expressing_cells
- confirms that genes lost to the 30-cell cutoff are predominantly low-expression (expected)

Diag 7: per-gene UTR raw counts vs lognorm expression

- x: log1p(mean UTR raw counts); y: mean lognorm expression
- cross-assay sanity: the two assays should agree per gene

---

### Result plots (12 per lineage)

Result 1: signed rho histogram by significance

- x: rho_W_expr; y: gene count; grey = padj>=0.05, firebrick = padj<0.05
- asymmetry between pos and neg sig tails is the directional bias signal

Result 2: |rho| histogram by significance

- x: |rho_W_expr|; y: gene count; same colour scheme as Result 1

Result 3: volcano (with ASD overlay)

- x: rho_W_expr; y: -log10(padj_BH)
- point colour: blue (sig neg), red (sig pos), grey (NS)
- orange star overlay: ASD-flagged genes from `asd_significant_in_knoblich.csv` (top 5 ASD hits labeled with gene name)
- dashed lines: padj=0.05 horizontal, rho=0 vertical

Result 4: MA-style mean expression vs rho

- x: mean lognorm expression; y: rho_W_expr; colours as in Result 3
- tells whether highly expressed genes are systematically more or less coupled

Result 5: rho_W_expr by K (boxplot)

- x: K-bin (2, 3, 4, 5+); y: rho_W_expr distribution
- if K dominates, conclusions are size-driven not biology-driven

Result 6: marginal vs within-celltype median rho

- x: marginal rho_W_expr over the whole lineage; y: median per-celltype rho
- diagonal = consistent within-state coupling
- red dots = "lineage-driven" outliers (large marginal but tiny within-celltype median)

Result 7: heatmap of top 50 pos + top 50 neg rho_W_expr

- rows: 100 genes (50 pos then 50 neg, separated by a horizontal black line); ASD-flagged genes prefixed with `*`
- columns: lineage celltypes
- colour: per-celltype rho (RdBu_r, centred at 0; blue = neg, red = pos)

Result 8a / 8b: top-12 positive / negative scatter panels

- 4x3 grid; per panel: x = W_c, y = lognorm expression; colour = celltype
- title: gene name (with "(ASD)" suffix if ASD-flagged), rho, padj

Result 9: rho_W_expr vs rho_W_pseudotime (cross-comparison)

- x: rho_W against pseudotime (from `effect_sizes/<lineage>/rho_W_per_gene.csv`)
- y: this notebook's rho_W_expr (vs lognorm expression)
- orange stars: ASD-flagged genes; title reports Pearson r between the two W-anchored rho measurements

Result 10: rho_W_expr vs delta_psi (cross-comparison)

- x: delta_psi from the composite ranking (signed Wasserstein-1 net shift along pseudotime; positive = lengthening)
- y: rho_W_expr
- positive correlation = lengthening genes also up-regulate

Result 11: rho_W_expr stratified by pseudotime LRT-padj

- x: two violins, LRT NS vs LRT sig (from `dirmult_results/.../dirmult_regression_optimized2_joint_df5_*.csv`); y: rho_W_expr distribution
- tests whether APA-dynamic genes (LRT-sig) are disproportionately expression-coupled

---

### Treatment cross-comparison plots (4 per lineage)

Treat 1: rho_induced vs rho_noninduced (per-gene scatter)

- x: rho_W_expr computed using only non-induced cells in this lineage
- y: rho_W_expr computed using only induced cells in this lineage
- purple overlay: treatment-LRT-significant genes (DRIMSeq `adj_pvalue` < 0.05 from `dirmult_treatment_results/.../drimseq_gene_<lineage>_treatment_main.csv`)
- orange stars: ASD-flagged genes
- diagonal = no treatment-driven coupling shift; off-diagonal = treatment-altered coupling

Treat 2: delta_rho vs delta_psi_treatment

- x: delta_psi_treatment (signed net UTR shift between induced and non-induced, from the treatment DRIMSeq CSV)
- y: delta_rho_treatment (rho_induced - rho_noninduced)
- a positive correlation here would say "genes that lengthen with treatment also up-couple their W-expression relationship with treatment"

Treat 3: rho_W_expr violin split by treatment LRT-sig

- x: two violins, treatment NS vs treatment sig; y: marginal rho_W_expr
- tests whether the treatment-dynamic genes are also disproportionately W-expression coupled

Treat 4: convergent exemplars

- six genes that pass treatment-LRT-sig and |delta_rho_treatment|>0.2 and |rho_W_expr|>0.3
- per panel: x = W_c, y = lognorm expression; blue dots = non-induced cells, red dots = induced cells
- if no such genes exist, the plot is skipped (see the main results section)

---

### Cross-lineage plot (final)

EXC vs INH rho_W_expr

- x: rho_W_expr from the EXC notebook output; y: rho_W_expr from the INH notebook output
- diagonal = lineage-consistent coupling; off-diagonal = lineage-specific
- orange stars: ASD-flagged genes
- annotated labels: top 10 |rho| in EXC and top 10 |rho| in INH (union)
- title reports Pearson r between the two lineages


In [ ]:
def _gene_lookup(gene_meta, gid):
    for tup in gene_meta:
        if tup[0] == gid:
            return tup
    return None


def diagnostic_plots(
    df_gene,
    lineage_label,
    lineage_suffix,
    ct_present,
    X_utr_lin,
    X_expr_lin,
    induced_lin,
    log_lib_lin,
    celltype_arr,
):
    """7-diagnostic panel: validate test design per lineage."""
    sig = df_gene["padj_BH"] < SIG_THR

    # Diag 1: n_expressing_cells histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(
        df_gene["n_expressing_cells"].dropna(),
        bins=60,
        color="#666",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.axvline(
        MIN_EXPRESSING_CELLS,
        color=COL_LONG,
        ls="--",
        lw=1,
        label=f"min={MIN_EXPRESSING_CELLS}",
    )
    ax.set_xlabel("n_expressing_cells")
    ax.set_ylabel("genes")
    ax.legend()
    ax.set_title(f"Diag 1: n_expressing_cells per gene [{lineage_label}]")
    savefig(f"diag1_n_expressing_cells_hist_{lineage_suffix}")
    plt.show()

    # Diag 2: CDF of K
    retained = df_gene[df_gene["rho_W_expr"].notna()]
    ks = np.sort(retained["n_sites"].values)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ks, np.linspace(0, 1, len(ks)), color="#333")
    ax.set_xlabel("K (sites per gene)")
    ax.set_ylabel("cumulative fraction retained")
    ax.set_title(f"Diag 2: CDF of K [{lineage_label}]")
    savefig(f"diag2_K_cdf_{lineage_suffix}")
    plt.show()

    # Diag 3: 6 example genes
    ct_palette = dict(zip(ct_present, cm.tab10.colors[: len(ct_present)]))
    top_cov = retained.sort_values("n_expressing_cells", ascending=False).head(3)
    low_cov = (
        retained[retained["n_expressing_cells"] >= MIN_EXPRESSING_CELLS]
        .sort_values("n_expressing_cells", ascending=True)
        .head(3)
    )
    example_genes = list(top_cov["gene_id"]) + list(low_cov["gene_id"])
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, gid in zip(axes.flat, example_genes):
        tup = _gene_lookup(gene_meta, gid)
        if tup is None:
            ax.axis("off")
            continue
        _, gname, sidx, K, ranks, eidx = tup
        sub = X_utr_lin[:, sidx]
        tot = np.asarray(sub.sum(axis=1)).ravel()
        m = tot >= MIN_GENE_COUNTS
        W_c = (sub[m].toarray() / tot[m, None]) @ ranks
        ec = X_expr_lin[:, eidx]
        ec = np.asarray(ec.todense()).ravel() if issparse(ec) else ec
        ec = ec[m]
        cts = celltype_arr[m]
        for ct in ct_present:
            sel = cts == ct
            if sel.sum():
                ax.scatter(
                    W_c[sel], ec[sel], s=4, alpha=0.5, color=ct_palette[ct], label=ct
                )
        rho_row = df_gene[df_gene["gene_id"] == gid].iloc[0]
        ax.set_title(
            f"{gname} (K={K}, n={int(rho_row.n_expressing_cells)}, rho={rho_row.rho_W_expr:.2f})",
            fontsize=10,
        )
        ax.set_xlabel("W_c")
        ax.set_ylabel("lognorm expression")
    axes[0, 0].legend(fontsize=6, ncol=2)
    fig.suptitle(
        f"Diag 3: 6 example genes (top: high-cov, bottom: low-cov) [{lineage_label}]"
    )
    plt.tight_layout()
    savefig(f"diag3_example_genes_scatter_{lineage_suffix}")
    plt.show()

    # Diag 4: rho_W_libsize
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(
        df_gene["rho_W_libsize"].dropna(),
        bins=60,
        color="#777",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.axvline(0, color="black", ls="--", lw=0.6)
    ax.set_xlabel("Spearman(W_c, log(total_counts))")
    ax.set_ylabel("genes")
    ax.set_title(
        f"Diag 4: library-size confounding [{lineage_label}] (peaks at +/-1 = warning)"
    )
    savefig(f"diag4_W_vs_libsize_{lineage_suffix}")
    plt.show()

    # Diag 5: W per celltype pooled top-100
    hi = df_gene[
        (df_gene["abs_rho_W_expr"] > RHO_VIS_THR) & (df_gene["padj_BH"] < SIG_THR)
    ].head(100)
    if len(hi):
        W_pool_lst, ct_pool_lst = [], []
        for gid in hi["gene_id"]:
            tup = _gene_lookup(gene_meta, gid)
            if tup is None:
                continue
            _, _, sidx, K, ranks, _ = tup
            sub = X_utr_lin[:, sidx]
            tot = np.asarray(sub.sum(axis=1)).ravel()
            m = tot >= MIN_GENE_COUNTS
            W_c = (sub[m].toarray() / tot[m, None]) @ ranks
            W_pool_lst.append(W_c)
            ct_pool_lst.append(celltype_arr[m])
        W_pool = np.concatenate(W_pool_lst)
        ct_pool = np.concatenate(ct_pool_lst)
        df_pool = pd.DataFrame({"W": W_pool, "celltype": ct_pool})
        df_pool["celltype"] = pd.Categorical(df_pool["celltype"], categories=ct_present)
        data_by_ct = [
            df_pool.loc[df_pool.celltype == ct, "W"].values for ct in ct_present
        ]
        fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(ct_present) + 2), 4))
        ax.violinplot(data_by_ct, showmedians=True)
        ax.set_xticks(range(1, len(ct_present) + 1))
        ax.set_xticklabels(ct_present, rotation=45)
        ax.set_ylabel(f"W_c pooled over top-{len(hi)} sig genes")
        ax.set_title(f"Diag 5: W per celltype [{lineage_label}]")
        savefig(f"diag5_W_per_celltype_violin_{lineage_suffix}")
        plt.show()
    else:
        print(f"Diag 5 skipped for {lineage_label}")

    # Diag 6
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        df_gene["mean_lognorm_expr"],
        df_gene["n_expressing_cells"],
        s=3,
        alpha=0.3,
        color="#444",
        rasterized=True,
    )
    ax.set_xlabel("mean lognorm expression")
    ax.set_ylabel("n_expressing_cells")
    ax.set_title(f"Diag 6: detection vs expression [{lineage_label}]")
    savefig(f"diag6_detection_vs_expression_{lineage_suffix}")
    plt.show()

    # Diag 7: per-gene UTR raw mean vs lognorm mean
    gene_utr_mean = np.zeros(len(gene_meta))
    gid_to_idx = {tup[0]: i for i, tup in enumerate(gene_meta)}
    for i, tup in enumerate(gene_meta):
        sidx = tup[2]
        gene_utr_mean[i] = float(
            np.asarray(X_utr_lin[:, sidx].sum(axis=1)).ravel().mean()
        )
    rho_row_idx = df_gene.set_index("gene_id")["mean_lognorm_expr"].to_dict()
    gene_expr_mean = np.array([rho_row_idx.get(tup[0], np.nan) for tup in gene_meta])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        np.log1p(gene_utr_mean),
        gene_expr_mean,
        s=3,
        alpha=0.3,
        rasterized=True,
        color="#222",
    )
    ax.set_xlabel("log1p mean UTR raw counts")
    ax.set_ylabel("mean lognorm expression")
    ax.set_title(f"Diag 7: UTR raw vs lognorm per gene [{lineage_label}]")
    savefig(f"diag7_utr_vs_lognorm_per_gene_{lineage_suffix}")
    plt.show()

    return ct_palette

In [ ]:
def result_plots(
    df_gene,
    lineage_label,
    lineage_suffix,
    ct_present,
    ct_palette,
    X_utr_lin,
    X_expr_lin,
    celltype_arr,
):
    """11 result plots (res1-res11) per lineage."""
    sig = df_gene["padj_BH"] < SIG_THR

    # Result 1
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(
        df_gene.loc[~sig, "rho_W_expr"].dropna(),
        bins=60,
        alpha=0.5,
        color=COL_NS,
        label=f"padj >= {SIG_THR}",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.hist(
        df_gene.loc[sig, "rho_W_expr"].dropna(),
        bins=60,
        alpha=0.7,
        color="firebrick",
        label=f"padj < {SIG_THR}",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.axvline(0, color="black", ls="--", lw=0.6)
    ax.set_xlabel("rho_W_expr")
    ax.set_ylabel("genes")
    ax.legend()
    ax.set_title(f"Result 1: rho_W_expr by significance [{lineage_label}]")
    savefig(f"res1_rho_signed_hist_{lineage_suffix}")
    plt.show()

    # Result 2
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(
        df_gene.loc[~sig, "abs_rho_W_expr"].dropna(),
        bins=40,
        alpha=0.5,
        color=COL_NS,
        label=f"padj >= {SIG_THR}",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.hist(
        df_gene.loc[sig, "abs_rho_W_expr"].dropna(),
        bins=40,
        alpha=0.7,
        color="firebrick",
        label=f"padj < {SIG_THR}",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.set_xlabel("|rho_W_expr|")
    ax.set_ylabel("genes")
    ax.legend()
    ax.set_title(f"Result 2: |rho_W_expr| [{lineage_label}]")
    savefig(f"res2_abs_rho_hist_{lineage_suffix}")
    plt.show()

    # Result 3: volcano + ASD overlay
    fig, ax = plt.subplots(figsize=(7, 6))
    y_volc = -np.log10(df_gene["padj_BH"].clip(lower=1e-300))
    col_v = np.where(
        sig & (df_gene["rho_W_expr"] > 0),
        COL_LONG,
        np.where(sig & (df_gene["rho_W_expr"] < 0), COL_SHORT, COL_NS),
    )
    ax.scatter(df_gene["rho_W_expr"], y_volc, c=col_v, s=5, alpha=0.6, rasterized=True)
    asd_mask = df_gene["asd_any"].fillna(False)
    ax.scatter(
        df_gene.loc[asd_mask, "rho_W_expr"],
        y_volc[asd_mask],
        s=30,
        marker="*",
        color=COL_ASD,
        edgecolor="black",
        linewidth=0.3,
        alpha=0.95,
        label=f"ASD-flagged (n={int(asd_mask.sum())})",
        zorder=5,
    )
    ax.axhline(-np.log10(SIG_THR), color="grey", ls="--", lw=0.5)
    ax.axvline(0, color="grey", ls="--", lw=0.5)
    n_pos = int(((df_gene["rho_W_expr"] > 0) & sig).sum())
    n_neg = int(((df_gene["rho_W_expr"] < 0) & sig).sum())
    ax.set_xlabel("rho_W_expr")
    ax.set_ylabel("-log10(padj_BH)")
    ax.set_title(
        f"Result 3: volcano [{lineage_label}] (pos sig={n_pos}, neg sig={n_neg})"
    )
    # label top-5 ASD hits
    top_asd = (
        df_gene[asd_mask]
        .dropna(subset=["rho_W_expr"])
        .sort_values("abs_rho_W_expr", ascending=False)
        .head(5)
    )
    for _, r in top_asd.iterrows():
        ax.annotate(
            r.gene_name,
            (r.rho_W_expr, -np.log10(max(r.padj_BH, 1e-300))),
            fontsize=7,
            alpha=0.9,
        )
    ax.legend(fontsize=7, loc="upper left")
    savefig(f"res3_volcano_{lineage_suffix}")
    plt.show()

    # Result 4: MA
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        df_gene["mean_lognorm_expr"],
        df_gene["rho_W_expr"],
        c=col_v,
        s=5,
        alpha=0.4,
        rasterized=True,
    )
    ax.axhline(0, color="black", lw=0.5, ls="--")
    ax.set_xlabel("mean lognorm expression")
    ax.set_ylabel("rho_W_expr")
    ax.set_title(f"Result 4: rho vs mean expression [{lineage_label}]")
    savefig(f"res4_MA_{lineage_suffix}")
    plt.show()

    # Result 5
    df_K = df_gene.copy()
    df_K["K_bin"] = np.where(df_K["n_sites"] >= 5, "5+", df_K["n_sites"].astype(str))
    order = ["2", "3", "4", "5+"]
    data_K = [df_K[df_K["K_bin"] == k]["rho_W_expr"].dropna().values for k in order]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.boxplot(data_K, tick_labels=order, showfliers=False)
    ax.axhline(0, color="grey", ls="--", lw=0.5)
    ax.set_xlabel("n_sites K")
    ax.set_ylabel("rho_W_expr")
    ax.set_title(f"Result 5: rho_W_expr by K [{lineage_label}]")
    savefig(f"res5_rho_by_K_{lineage_suffix}")
    plt.show()

    # Result 6: marginal vs within-CT
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        df_gene["rho_W_expr"],
        df_gene["rho_W_expr_celltype_median"],
        s=4,
        alpha=0.4,
        rasterized=True,
        color="#333",
    )
    ax.plot([-1, 1], [-1, 1], "k--", lw=0.6)
    ax.axhline(0, color="grey", lw=0.4)
    ax.axvline(0, color="grey", lw=0.4)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel("Marginal rho_W_expr")
    ax.set_ylabel("Within-celltype median rho")
    disc = (df_gene["rho_W_expr"].abs() > RHO_VIS_THR) & (
        df_gene["rho_W_expr_celltype_median"].abs() < 0.1
    )
    ax.scatter(
        df_gene.loc[disc, "rho_W_expr"],
        df_gene.loc[disc, "rho_W_expr_celltype_median"],
        s=12,
        color=COL_LONG,
        alpha=0.7,
        label=f"lineage-driven (n={int(disc.sum())})",
    )
    ax.legend()
    ax.set_title(f"Result 6: marginal vs within-CT median rho [{lineage_label}]")
    savefig(f"res6_marginal_vs_within_ct_{lineage_suffix}")
    plt.show()

    # Result 7: heatmap top 50 pos + 50 neg
    top_pos = (
        df_gene.dropna(subset=["rho_W_expr"])
        .sort_values("rho_W_expr", ascending=False)
        .head(50)
    )
    top_neg = (
        df_gene.dropna(subset=["rho_W_expr"])
        .sort_values("rho_W_expr", ascending=True)
        .head(50)
    )
    panel = pd.concat([top_pos, top_neg], axis=0)
    heat = np.full((len(panel), len(ct_present)), np.nan)
    for r_i, gid in enumerate(panel["gene_id"]):
        tup = _gene_lookup(gene_meta, gid)
        if tup is None:
            continue
        _, _, sidx, K, ranks, eidx = tup
        sub = X_utr_lin[:, sidx]
        tot = np.asarray(sub.sum(axis=1)).ravel()
        m = tot >= MIN_GENE_COUNTS
        W_c = (sub[m].toarray() / tot[m, None]) @ ranks
        ec = X_expr_lin[:, eidx]
        ec = np.asarray(ec.todense()).ravel() if issparse(ec) else ec
        ec = ec[m]
        cts = celltype_arr[m]
        for c_i, ct in enumerate(ct_present):
            mm = cts == ct
            if (
                int(mm.sum()) >= MIN_CELLS_PER_CT
                and W_c[mm].std() > 0
                and ec[mm].std() > 0
            ):
                rho_, _ = spearmanr(W_c[mm], ec[mm])
                heat[r_i, c_i] = rho_
    fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(ct_present) + 1), 14))
    im = ax.imshow(heat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    asd_in_panel = panel["asd_any"].fillna(False).values
    ylabels = []
    for nm, asd in zip(panel["gene_name"].values, asd_in_panel):
        ylabels.append(f"* {nm}" if asd else nm)
    ax.set_yticks(range(len(panel)))
    ax.set_yticklabels(ylabels, fontsize=6)
    ax.set_xticks(range(len(ct_present)))
    ax.set_xticklabels(ct_present, rotation=45)
    ax.axhline(len(top_pos) - 0.5, color="black", lw=0.8)
    ax.set_title(
        f"Result 7: top 50 pos + top 50 neg per-celltype rho [{lineage_label}]\n* = ASD-flagged"
    )
    fig.colorbar(im, ax=ax, label="rho")
    savefig(f"res7_heatmap_top100_{lineage_suffix}")
    plt.show()

    # Result 8a, 8b
    def panel_scatter(gene_ids, title, fname):
        fig, axes = plt.subplots(4, 3, figsize=(14, 14))
        for ax, gid in zip(axes.flat, gene_ids):
            tup = _gene_lookup(gene_meta, gid)
            if tup is None:
                ax.axis("off")
                continue
            _, gname, sidx, K, ranks, eidx = tup
            sub = X_utr_lin[:, sidx]
            tot = np.asarray(sub.sum(axis=1)).ravel()
            m = tot >= MIN_GENE_COUNTS
            W_c = (sub[m].toarray() / tot[m, None]) @ ranks
            ec = X_expr_lin[:, eidx]
            ec = np.asarray(ec.todense()).ravel() if issparse(ec) else ec
            ec = ec[m]
            cts = celltype_arr[m]
            for ct in ct_present:
                sel = cts == ct
                if sel.sum():
                    ax.scatter(
                        W_c[sel],
                        ec[sel],
                        s=3,
                        alpha=0.4,
                        color=ct_palette[ct],
                        label=ct,
                    )
            row = df_gene[df_gene["gene_id"] == gid].iloc[0]
            asd_tag = " (ASD)" if row.get("asd_any", False) else ""
            ax.set_title(
                f"{gname}{asd_tag}  rho={row.rho_W_expr:.2f}  padj={row.padj_BH:.1e}",
                fontsize=9,
            )
            ax.set_xlabel("W_c")
            ax.set_ylabel("lognorm expr")
        axes[0, 0].legend(fontsize=6, ncol=2)
        fig.suptitle(title)
        plt.tight_layout()
        savefig(fname)
        plt.show()

    panel_scatter(
        top_pos.head(12)["gene_id"].tolist(),
        f"Result 8a: top 12 POS rho [{lineage_label}]",
        f"res8a_top12_positive_{lineage_suffix}",
    )
    panel_scatter(
        top_neg.head(12)["gene_id"].tolist(),
        f"Result 8b: top 12 NEG rho [{lineage_label}]",
        f"res8b_top12_negative_{lineage_suffix}",
    )

    # Result 9: rho_W_expr vs rho_W_pseudotime
    m9 = df_gene.dropna(subset=["rho_W_expr", "rho_W_pseudotime"])
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        m9["rho_W_pseudotime"],
        m9["rho_W_expr"],
        s=4,
        alpha=0.4,
        color="#333",
        rasterized=True,
    )
    ax.axhline(0, color="grey", lw=0.4)
    ax.axvline(0, color="grey", lw=0.4)
    asd9 = m9["asd_any"].fillna(False)
    ax.scatter(
        m9.loc[asd9, "rho_W_pseudotime"],
        m9.loc[asd9, "rho_W_expr"],
        s=30,
        marker="*",
        color=COL_ASD,
        edgecolor="black",
        linewidth=0.3,
        alpha=0.95,
        label=f"ASD (n={int(asd9.sum())})",
        zorder=5,
    )
    r_p = m9[["rho_W_pseudotime", "rho_W_expr"]].corr().iloc[0, 1]
    ax.set_xlabel("rho_W_pseudotime")
    ax.set_ylabel("rho_W_expr")
    ax.set_title(
        f"Result 9: rho_W_expr vs rho_W_pseudotime [{lineage_label}]; pearson r={r_p:.3f}"
    )
    ax.legend(fontsize=7)
    savefig(f"res9_vs_pseudotime_rho_{lineage_suffix}")
    plt.show()

    # Result 10: rho_W_expr vs delta_psi
    m10 = df_gene.dropna(subset=["rho_W_expr", "delta_psi"])
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        m10["delta_psi"],
        m10["rho_W_expr"],
        s=4,
        alpha=0.4,
        color="#333",
        rasterized=True,
    )
    ax.axhline(0, color="grey", lw=0.4)
    ax.axvline(0, color="grey", lw=0.4)
    asd10 = m10["asd_any"].fillna(False)
    ax.scatter(
        m10.loc[asd10, "delta_psi"],
        m10.loc[asd10, "rho_W_expr"],
        s=30,
        marker="*",
        color=COL_ASD,
        edgecolor="black",
        linewidth=0.3,
        alpha=0.95,
        label=f"ASD (n={int(asd10.sum())})",
        zorder=5,
    )
    r10 = m10[["delta_psi", "rho_W_expr"]].corr().iloc[0, 1]
    ax.set_xlabel("delta_psi (pseudotime)")
    ax.set_ylabel("rho_W_expr")
    ax.set_title(
        f"Result 10: rho_W_expr vs delta_psi [{lineage_label}]; pearson r={r10:.3f}"
    )
    ax.legend(fontsize=7)
    savefig(f"res10_vs_delta_psi_{lineage_suffix}")
    plt.show()

    # Result 11: by pseudotime LRT padj
    lrt_p = df_gene["lrt_padj_pseudotime"]
    lrt_sig = (lrt_p < SIG_THR) & lrt_p.notna()
    has_lrt = lrt_p.notna() & df_gene["rho_W_expr"].notna()
    data = [
        df_gene.loc[has_lrt & ~lrt_sig, "rho_W_expr"].values,
        df_gene.loc[has_lrt & lrt_sig, "rho_W_expr"].values,
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    if len(data[1]) > 0:
        ax.violinplot(data, showmedians=True)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(
            [f"LRT NS (n={len(data[0])})", f"LRT sig (n={len(data[1])})"]
        )
    else:
        print(f"Result 11 skipped for {lineage_label}: no LRT-sig genes after merge")
    ax.set_ylabel("rho_W_expr")
    ax.axhline(0, color="grey", lw=0.4)
    ax.set_title(f"Result 11: rho_W_expr by LRT-padj_pseudotime [{lineage_label}]")
    savefig(f"res11_by_lrt_sig_pseudotime_{lineage_suffix}")
    plt.show()

In [ ]:
def treatment_plots(
    df_gene,
    lineage_label,
    lineage_suffix,
    ct_present,
    ct_palette,
    X_utr_lin,
    X_expr_lin,
    induced_lin,
    celltype_arr,
):
    """4 treatment cross-comparison plots."""
    # Treat 1: rho_induced vs rho_noninduced
    m1 = df_gene.dropna(subset=["rho_induced", "rho_noninduced"])
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        m1["rho_noninduced"],
        m1["rho_induced"],
        s=4,
        alpha=0.4,
        color="#333",
        rasterized=True,
    )
    ax.plot([-1, 1], [-1, 1], "k--", lw=0.6)
    ax.axhline(0, color="grey", lw=0.4)
    ax.axvline(0, color="grey", lw=0.4)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    treat_sig = (m1["lrt_padj_treatment"] < SIG_THR) & m1["lrt_padj_treatment"].notna()
    if treat_sig.any():
        ax.scatter(
            m1.loc[treat_sig, "rho_noninduced"],
            m1.loc[treat_sig, "rho_induced"],
            s=18,
            color=COL_TREAT,
            alpha=0.85,
            edgecolor="black",
            linewidth=0.2,
            label=f"treatment-LRT-sig (n={int(treat_sig.sum())})",
            zorder=4,
        )
    asd_m1 = m1["asd_any"].fillna(False)
    if asd_m1.any():
        ax.scatter(
            m1.loc[asd_m1, "rho_noninduced"],
            m1.loc[asd_m1, "rho_induced"],
            s=30,
            marker="*",
            color=COL_ASD,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.95,
            label=f"ASD (n={int(asd_m1.sum())})",
            zorder=5,
        )
    ax.set_xlabel("rho_W_expr (non-induced)")
    ax.set_ylabel("rho_W_expr (induced)")
    ax.set_title(f"Treat 1: rho_induced vs rho_noninduced [{lineage_label}]")
    ax.legend(fontsize=7)
    savefig(f"treat1_rho_induced_vs_noninduced_{lineage_suffix}")
    plt.show()

    # Treat 2: delta_rho vs delta_psi_treatment
    m2 = df_gene.dropna(subset=["delta_rho_treatment", "delta_psi_treatment"])
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        m2["delta_psi_treatment"],
        m2["delta_rho_treatment"],
        s=4,
        alpha=0.4,
        color="#333",
        rasterized=True,
    )
    ax.axhline(0, color="grey", lw=0.4)
    ax.axvline(0, color="grey", lw=0.4)
    treat_sig2 = (m2["lrt_padj_treatment"] < SIG_THR) & m2["lrt_padj_treatment"].notna()
    if treat_sig2.any():
        ax.scatter(
            m2.loc[treat_sig2, "delta_psi_treatment"],
            m2.loc[treat_sig2, "delta_rho_treatment"],
            s=18,
            color=COL_TREAT,
            alpha=0.85,
            edgecolor="black",
            linewidth=0.2,
            label=f"treat-LRT-sig (n={int(treat_sig2.sum())})",
            zorder=4,
        )
    asd_m2 = m2["asd_any"].fillna(False)
    if asd_m2.any():
        ax.scatter(
            m2.loc[asd_m2, "delta_psi_treatment"],
            m2.loc[asd_m2, "delta_rho_treatment"],
            s=30,
            marker="*",
            color=COL_ASD,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.95,
            label=f"ASD (n={int(asd_m2.sum())})",
            zorder=5,
        )
    ax.set_xlabel("delta_psi_treatment (induced - noninduced)")
    ax.set_ylabel("delta_rho_treatment (rho_induced - rho_noninduced)")
    r2 = m2[["delta_psi_treatment", "delta_rho_treatment"]].corr().iloc[0, 1]
    ax.set_title(
        f"Treat 2: delta_rho vs delta_psi treatment [{lineage_label}]; r={r2:.3f}"
    )
    ax.legend(fontsize=7)
    savefig(f"treat2_delta_rho_vs_delta_psi_treatment_{lineage_suffix}")
    plt.show()

    # Treat 3: rho_W_expr violin by treatment-LRT-sig
    has_t = df_gene["lrt_padj_treatment"].notna() & df_gene["rho_W_expr"].notna()
    t_sig = (df_gene["lrt_padj_treatment"] < SIG_THR) & has_t
    data = [
        df_gene.loc[has_t & ~t_sig, "rho_W_expr"].values,
        df_gene.loc[has_t & t_sig, "rho_W_expr"].values,
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    if len(data[1]) >= 3:
        ax.violinplot(data, showmedians=True)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(
            [f"treat NS (n={len(data[0])})", f"treat sig (n={len(data[1])})"]
        )
        ax.set_ylabel("rho_W_expr (marginal)")
        ax.axhline(0, color="grey", lw=0.4)
    else:
        ax.text(
            0.5,
            0.5,
            f"only {len(data[1])} treatment-sig genes - skipping",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_title(f"Treat 3: rho_W_expr by treatment LRT sig [{lineage_label}]")
    savefig(f"treat3_rho_W_expr_by_treatment_lrt_sig_{lineage_suffix}")
    plt.show()

    # Treat 4: convergent exemplars (top 6)
    conv = df_gene[
        (df_gene["lrt_padj_treatment"] < SIG_THR)
        & (df_gene["abs_delta_rho_treatment"] > 0.2)
        & (df_gene["abs_rho_W_expr"] > RHO_VIS_THR)
    ].copy()
    conv = conv.sort_values("abs_delta_rho_treatment", ascending=False).head(6)
    print(
        f"Treat 4: {len(conv)} convergent exemplars (treat-sig + |delta_rho|>0.2 + |rho|>0.3)"
    )
    if len(conv) == 0:
        print(f"Treat 4 skipped for {lineage_label}")
        return
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, (_, row) in zip(axes.flat, conv.iterrows()):
        tup = _gene_lookup(gene_meta, row["gene_id"])
        if tup is None:
            ax.axis("off")
            continue
        _, gname, sidx, K, ranks, eidx = tup
        sub = X_utr_lin[:, sidx]
        tot = np.asarray(sub.sum(axis=1)).ravel()
        m = tot >= MIN_GENE_COUNTS
        W_c = (sub[m].toarray() / tot[m, None]) @ ranks
        ec = X_expr_lin[:, eidx]
        ec = np.asarray(ec.todense()).ravel() if issparse(ec) else ec
        ec = ec[m]
        ind = induced_lin[m]
        ax.scatter(
            W_c[~ind],
            ec[~ind],
            s=3,
            alpha=0.35,
            color=COL_SHORT,
            label=f"non-induced (n={(~ind).sum()})",
            rasterized=True,
        )
        ax.scatter(
            W_c[ind],
            ec[ind],
            s=3,
            alpha=0.35,
            color=COL_LONG,
            label=f"induced (n={(ind).sum()})",
            rasterized=True,
        )
        asd_tag = " (ASD)" if row.get("asd_any", False) else ""
        ax.set_title(
            f"{gname}{asd_tag}\nrho_ind={row.rho_induced:.2f}  rho_noninfd={row.rho_noninduced:.2f}\ndelta_psi_treat={row.delta_psi_treatment:.3f}",
            fontsize=8,
        )
        ax.set_xlabel("W_c")
        ax.set_ylabel("lognorm expr")
        ax.legend(fontsize=6)
    fig.suptitle(
        f"Treat 4: convergent exemplars (treat-sig + |delta_rho|>0.2 + |rho|>0.3) [{lineage_label}]"
    )
    plt.tight_layout()
    savefig(f"treat4_convergent_exemplars_{lineage_suffix}")
    plt.show()

## Diagnostic + result + treatment plots for EXC


In [ ]:
# Build the lineage-specific arrays once for plotting
def _lineage_arrays(lineage_name, pt_col):
    obs = adata_utr.obs
    mask = (obs["lineage"] == lineage_name) & obs[pt_col].notna()
    cell_idx = np.where(mask.values)[0]
    X_utr_lin = X_utr_csc[cell_idx, :].tocsc()
    X_expr_lin = X_expr_csr[cell_idx, :]
    induced_lin = obs["induced"].values[cell_idx]
    celltype_arr = obs["celltype"].iloc[cell_idx].astype(str).values
    log_lib_lin = log_total_counts_all[cell_idx]
    ct_series = pd.Series(celltype_arr)
    ct_present = sorted(
        [c for c in ct_series.unique() if (ct_series == c).sum() >= MIN_CELLS_PER_CT]
    )
    return X_utr_lin, X_expr_lin, induced_lin, celltype_arr, log_lib_lin, ct_present


X_utr_EXC, X_expr_EXC, induced_EXC, ct_arr_EXC, log_lib_EXC, ct_present_EXC = (
    _lineage_arrays("Excitatory neurons", "dpt_pseudotime_excitatory")
)
print(f"EXC arrays: {X_utr_EXC.shape}, celltypes: {ct_present_EXC}")

ct_palette_EXC = diagnostic_plots(
    df_EXC,
    "Excitatory",
    "EXC",
    ct_present_EXC,
    X_utr_EXC,
    X_expr_EXC,
    induced_EXC,
    log_lib_EXC,
    ct_arr_EXC,
)
result_plots(
    df_EXC,
    "Excitatory",
    "EXC",
    ct_present_EXC,
    ct_palette_EXC,
    X_utr_EXC,
    X_expr_EXC,
    ct_arr_EXC,
)
treatment_plots(
    df_EXC,
    "Excitatory",
    "EXC",
    ct_present_EXC,
    ct_palette_EXC,
    X_utr_EXC,
    X_expr_EXC,
    induced_EXC,
    ct_arr_EXC,
)

## Diagnostic + result + treatment plots for INH


In [ ]:
X_utr_INH, X_expr_INH, induced_INH, ct_arr_INH, log_lib_INH, ct_present_INH = (
    _lineage_arrays("Inhibitory neurons", "dpt_pseudotime_inhibitory")
)
print(f"INH arrays: {X_utr_INH.shape}, celltypes: {ct_present_INH}")

ct_palette_INH = diagnostic_plots(
    df_INH,
    "Inhibitory",
    "INH",
    ct_present_INH,
    X_utr_INH,
    X_expr_INH,
    induced_INH,
    log_lib_INH,
    ct_arr_INH,
)
result_plots(
    df_INH,
    "Inhibitory",
    "INH",
    ct_present_INH,
    ct_palette_INH,
    X_utr_INH,
    X_expr_INH,
    ct_arr_INH,
)
treatment_plots(
    df_INH,
    "Inhibitory",
    "INH",
    ct_present_INH,
    ct_palette_INH,
    X_utr_INH,
    X_expr_INH,
    induced_INH,
    ct_arr_INH,
)

## Cross-lineage comparison

Per gene, plot EXC `rho_W_expr` vs INH `rho_W_expr`. Diagonal = lineage-
consistent coupling; off-diagonal = lineage-specific. ASD-flagged genes are
highlighted; top 10 |rho_EXC| and top 10 |rho_INH| are labeled.


In [ ]:
m_cross = (
    df_EXC[["gene_id", "gene_name", "rho_W_expr", "asd_any"]]
    .rename(columns={"rho_W_expr": "rho_EXC"})
    .merge(
        df_INH[["gene_id", "rho_W_expr"]].rename(columns={"rho_W_expr": "rho_INH"}),
        on="gene_id",
        how="inner",
    )
)
m_cross = m_cross.dropna(subset=["rho_EXC", "rho_INH"])
print(f"cross-lineage merged: {len(m_cross)} genes with rho in both lineages")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    m_cross["rho_EXC"],
    m_cross["rho_INH"],
    s=4,
    alpha=0.4,
    color="#333",
    rasterized=True,
)
ax.plot([-1, 1], [-1, 1], "k--", lw=0.6)
ax.axhline(0, color="grey", lw=0.4)
ax.axvline(0, color="grey", lw=0.4)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
asd_c = m_cross["asd_any"].fillna(False)
ax.scatter(
    m_cross.loc[asd_c, "rho_EXC"],
    m_cross.loc[asd_c, "rho_INH"],
    s=30,
    marker="*",
    color=COL_ASD,
    edgecolor="black",
    linewidth=0.3,
    alpha=0.95,
    label=f"ASD (n={int(asd_c.sum())})",
    zorder=5,
)
# label top 10 by |rho_EXC| and top 10 by |rho_INH|
m_cross["abs_exc"] = m_cross["rho_EXC"].abs()
m_cross["abs_inh"] = m_cross["rho_INH"].abs()
labels = set()
for sub_df in (m_cross.nlargest(10, "abs_exc"), m_cross.nlargest(10, "abs_inh")):
    for _, r in sub_df.iterrows():
        labels.add(r["gene_id"])
for gid in labels:
    r = m_cross[m_cross["gene_id"] == gid].iloc[0]
    ax.annotate(r["gene_name"], (r["rho_EXC"], r["rho_INH"]), fontsize=7, alpha=0.85)

r_pearson = m_cross[["rho_EXC", "rho_INH"]].corr().iloc[0, 1]
ax.set_xlabel("rho_W_expr (EXC)")
ax.set_ylabel("rho_W_expr (INH)")
ax.set_title(
    f"Cross-lineage: rho_W_expr EXC vs INH; n={len(m_cross)}, pearson r={r_pearson:.3f}"
)
ax.legend(fontsize=7)
savefig("crosslineage_rho_W_expr_EXC_vs_INH")
plt.show()

## Summary table

A condensed text summary of the main numbers per lineage and overall.


In [ ]:
def _summary(df, lineage_label):
    sig = df["padj_BH"] < SIG_THR
    print(f"--- {lineage_label} ---")
    print(f"  total genes: {len(df)} | non-NaN rho: {df['rho_W_expr'].notna().sum()}")
    print(
        f"  padj<0.05: {int(sig.sum())} (pos {int((sig & (df['rho_W_expr']>0)).sum())}, neg {int((sig & (df['rho_W_expr']<0)).sum())})"
    )
    print(f"  |rho|>0.3 AND padj<0.05: {int(((df['abs_rho_W_expr']>0.3) & sig).sum())}")
    print(
        f"  ASD-flagged with |rho|>0.3: {int(((df['abs_rho_W_expr']>0.3) & df['asd_any'].fillna(False)).sum())}"
    )
    print(
        f"  rho_W_libsize median: {df['rho_W_libsize'].median():.3f} | |>0.5|: {int((df['rho_W_libsize'].abs()>0.5).sum())}"
    )
    valid_corr = df.dropna(subset=["rho_W_expr", "rho_W_expr_celltype_median"])
    if len(valid_corr):
        r = valid_corr[["rho_W_expr", "rho_W_expr_celltype_median"]].corr().iloc[0, 1]
        print(f"  marginal vs within-CT median pearson r: {r:.3f}")
    # treatment
    n_with_treat = df["delta_rho_treatment"].notna().sum()
    print(
        f"  treatment-split: {n_with_treat} genes with both rho_induced and rho_noninduced"
    )
    n_tsig = (
        (df["lrt_padj_treatment"] < SIG_THR).sum()
        if "lrt_padj_treatment" in df.columns
        else 0
    )
    print(f"  treatment-LRT-sig (padj<0.05): {int(n_tsig)}")


_summary(df_EXC, "Excitatory")
_summary(df_INH, "Inhibitory")

## Main results and interpretation

### Main numbers

|                                                        | Excitatory (47k cells, 8 celltypes) | Inhibitory (24k cells, 6 celltypes) |
| ------------------------------------------------------ | ----------------------------------- | ----------------------------------- |
| Multi-site genes tested                                | 14,027                              | 14,027                              |
| Valid rho_W_expr (>=30 expressing cells)               | 13,491                              | 12,867                              |
| padj_BH < 0.05 (total / pos / neg)                     | 2,904 (1,874 / 1,030)               | 3,736 (1,273 / 2,463)               |
| rho >0.3 and padj<0.05                                 | 219                                 | 222                                 |
| Library-size confounding rho_W_libsize median (>0.5 n) | 0.012 (0)                           | 0.045 (5)                           |
| Marginal vs within-celltype rho Pearson r              | 0.849                               | 0.839                               |
| Merged with pseudotime LRT (sig at padj<0.05)          | 10,951 (912)                        | 9,334 (260)                         |
| Merged with treatment DRIMSeq LRT (sig)                | 6,997 (35)                          | 5,110 (201)                         |
| ASD-flagged genes present                              | 54                                  | 54                                  |

### Cross-lineage concordance

- 12,769 genes have a valid rho in both lineages; Pearson r(EXC, INH) = 0.640.
- 1,625 genes significant in both lineages (padj_BH<0.05); 94% same sign (1,533 concordant, 92 sign-flipped).
- 137 genes with |rho|>0.3 in both lineages (lineage-independent APA-expression coupling).

### Diagnostics

- Library-size confounding is negligible in both lineages, so W reflects UTR length rather than detection depth.
- Within-celltype consistency is high (r ~ 0.84 in both lineages): the per-gene coupling is a within-state property, not an artifact of celltype-level co-variation.

### Interpretation

Excitatory lineage: more positive than negative couplings among significant genes (1,874 vs 1,030, ~1.8x), i.e. longer-UTR isoforms go with higher expression of the same gene. The wildtype mouse SVZ data show the same bias (~2.3x).

Inhibitory lineage: the opposite asymmetry (2,463 negative vs 1,273 positive significant genes, ~1.9x), which is not seen in the mouse SVZ data. Possible explanations, not tested here:

1. GABAergic differentiation uses a post-transcriptional program in which long-UTR isoforms are destabilised (more miRNA target sites, AU-rich elements active in inhibitory progenitors).
2. The ventral (LGE/CGE-derived) inhibitory progenitor lineage uses a different set of RBPs (HuR, AUF1, FMRP, ...) whose target genes are biased toward the shortening direction.

Among genes significant in both lineages, 94% have the same sign, so the lineage-specific bias comes from lineage-specific sets of coupled genes rather than from sign flips within the shared set.

Treatment perturbation (CRISPR of 36 ASD-risk TFs):

- 35 EXC + 201 INH treatment-LRT-significant genes, mostly ribosomal and mitochondrial (NDUFA13, RPL26, RPS3A, EEF1A1, ARL6IP1, EID1, HSP90AA1, TIMM8B, SLIRP, RPS2, RAC1, CALM2, ...).
- 28 treatment-LRT-significant genes are significant in both lineages, mostly translation and OXPHOS genes.
- The largest |delta_rho_treatment| among treatment-significant genes is ~0.27 (RABEPK in INH). No gene passes the convergent-exemplar threshold (treatment-LRT-sig and |delta_rho|>0.2 and |rho_W_expr|>0.3). The perturbation shifts UTR usage but not the per-cell coupling between W and expression.

ASD genes (curated 63-gene reference set; 54 present in each lineage):

- Coupled with the same sign in both lineages: SMARCC2 (negative; strongest in INH, rho=-0.27, padj~1e-184), NRXN1 (negative), KCNQ2 (positive), JMJD1C (negative), OPHN1 (negative). Positive coupling in EXC: MEF2C, WWOX, CNTNAP2.
- For the negatively coupled genes (SMARCC2, NRXN1, JMJD1C), longer 3'UTR isoforms go with lower expression.
- The treatment hits are not SFARI-enriched as a group, and they overlap little with the ASD-coupled genes listed here.

The per-gene tables are plotted against the mouse data in fig. S2F (`figures/fig2/development/supp_4_xspecies_wexpr`).


## Caveats and limitations

- W is undefined for cells with zero UTR counts for a gene; sample size shrinks per gene.
- Detection-dependence between UTR counts and expression counts can create spurious positive correlation. The `rho_W_libsize` column quantifies the W vs log(total_counts) relationship per gene.
- Marginal rho mixes within- and across-cell-state variation. The within-celltype median rho is the within-state proxy; large discrepancies (large marginal, small median) flag "lineage-driven" coupling rather than per-state coupling.
- W collapses K-dimensional compositional information to a scalar. Use entropy_rho from the pseudotime correlation table for non-monotonic patterns.
- Correlation, not causation.
- Expression is recomputed from `.raw` via `normalize_total(1e4) + log1p`, without batch correction (the stored .X is HVG-restricted). Batch effects aligning with the lineage will inflate marginal rho; within-celltype median is partially robust since celltype calls were derived from batch-corrected data.
- Proportions are not smoothed, because graph smoothing would inflate rho via shared neighborhood structure.
- Two lineages (EXC and INH) are mutually exclusive at the cell level; per-gene results are not directly merged across them; the cross-lineage plot at the end is the only joint view.
- `induced` is constant within each sample (14 induced samples, 8 non-induced; super_batch 137 is 100% induced). Cell-level within-induced/within-non-induced Spearman is well-defined but the sample-level effective N is small. Treat the treatment-split rho as exploratory; the cross-comparison with the treatment DRIMSeq LRT (`lrt_padj_treatment`) is the safer outer signal.
- ASD highlight uses the curated `asd_significant_in_knoblich.csv` (63 SFARI+CHOOSE genes); the treatment hits are not SFARI-enriched as a group, so the ASD overlay flags exemplars and is not an enrichment test.
